In [ ]:
  // --- Dog System ---
  createDogMenu() {
    this.dogMenu = document.createElement('div');
    Object.assign(this.dogMenu.style, {
      position: 'absolute', top: '50%', left: '50%', transform: 'translate(-50%, -50%)',
      backgroundColor: 'rgba(0, 0, 0, 0.85)', padding: '20px', borderRadius: '10px',
      color: 'white', display: 'none', flexDirection: 'column', gap: '8px', zIndex: '2000',
      boxShadow: '0 0 15px rgba(255,255,255,0.2)'
    });
    this.dogMenu.innerHTML = `
      <h3 style="margin:0">Koira: <span id="dogName">Musti</span></h3>
      <div id="dogStats" style="font-size:12px; color:#ccc;"></div>
      <button id="btnDogPet">👋 Silitä</button>
      <button id="btnDogMode">⚔️ Tila: Seuraa</button>
      <div style="display:flex; gap:5px;">
        <button id="btnDogTakeMoney">💰 Ota rahat</button>
        <button id="btnDogGiveMoney">💵 Anna rahaa (10)</button>
      </div>
      <div>
        <strong>Repun sisältö (17 max):</strong>
        <div id="dogInv" style="display:flex; flex-wrap:wrap; gap:4px; max-width:200px; max-height:100px; overflow-y:auto; margin-top:5px; background:#222; padding:4px;"></div>
      </div>
      <button id="btnDogClose" style="margin-top:10px; background:#444;">Sulje</button>
    `;
    document.body.appendChild(this.dogMenu);

    this.dogMenu.querySelector('#btnDogPet').onclick = () => {
      this.dogHeartEffect();
      this.showToast('Silitit koiraa! ❤️');
    };
    this.dogMenu.querySelector('#btnDogMode').onclick = (e) => {
      const modes = ['follow', 'attack', 'fetch'];
      let idx = modes.indexOf(this.dogState.mode);
      this.dogState.mode = modes[(idx + 1) % modes.length];
      e.target.textContent = `⚔️ Tila: ${this.dogState.mode === 'follow' ? 'Seuraa' : this.dogState.mode === 'attack' ? 'Hyökkää' : 'Etsi rahaa'}`;
      this.refreshDogMenu();
      this.showToast(`Koiran tila: ${this.dogState.mode}`);
    };
    this.dogMenu.querySelector('#btnDogTakeMoney').onclick = () => {
      if (this.dogState.money > 0) {
        this.state.coins += this.dogState.money;
        this.updateUI();
        this.showToast(`Otit koiralta ${this.dogState.money} kolikkoa.`);
        this.dogState.money = 0;
        this.refreshDogMenu();
      } else {
        this.showToast('Koiralla ei ole rahaa.');
      }
    };
    this.dogMenu.querySelector('#btnDogGiveMoney').onclick = () => {
      if (this.state.coins >= 10) {
        this.state.coins -= 10;
        this.dogState.money += 10;
        this.updateUI();
        this.refreshDogMenu();
        this.showToast('Annoit koiralle 10 kolikkoa.');
      } else {
        this.showToast('Ei tarpeeksi rahaa!');
      }
    };
    this.dogMenu.querySelector('#btnDogClose').onclick = () => {
      this.hideDogMenu();
    };
  }

  showDogMenu() {
    if (!this.dogMenu) this.createDogMenu();
    this.refreshDogMenu();
    this.dogMenu.style.display = 'flex';
  }

  hideDogMenu() {
    if (this.dogMenu) this.dogMenu.style.display = 'none';
    this.resumeGame();
  }

  refreshDogMenu() {
    if (!this.dogMenu) return;
    const s = this.dogMenu.querySelector('#dogName');
    if (s) s.textContent = this.dogState.name;
    const stats = this.dogMenu.querySelector('#dogStats');
    if (stats) stats.textContent = `Rahaa: ${this.dogState.money} | Tila: ${this.dogState.mode}`;
    
    // Inventory
    const div = this.dogMenu.querySelector('#dogInv');
    div.innerHTML = '';
    this.dogState.inventory.forEach((item, i) => {
      const el = document.createElement('div');
      el.textContent = item;
      el.style.cssText = 'background:#444; padding:2px 4px; border-radius:3px; font-size:11px; cursor:pointer;';
      el.title = 'Klikkaa ottaaksesi';
      el.onclick = () => {
        // Move item to player
        this.dogState.inventory.splice(i, 1);
        if (this.inv[item] !== undefined) this.inv[item]++;
        this.updateInventoryUI();
        this.showToast(`Otit tavaran: ${item}`);
        this.refreshDogMenu();
      };
      div.appendChild(el);
    });
  }

  dogHeartEffect() {
    try {
      const h = this.add.text(this.dog.x, this.dog.y - 20, '❤️', {fontSize:'20px'});
      this.tweens.add({ targets: h, y: h.y - 30, alpha: 0, duration: 800, onComplete: ()=>h.destroy() });
      window.playSfx?.('powerup');
    } catch(e){}
  }

  toggleDogActive() {
    this.dogState.active = !this.dogState.active;
    this.dog.setActive(this.dogState.active).setVisible(this.dogState.active);
    if (this.dogState.active) {
      this.dog.setPosition(this.player.x, this.player.y);
      this.dog.setVisible(true);
      this.dog.body.enable = true;
      this.showToast('Koira aktivoitu! 🐶');
    } else {
      this.dog.setVisible(false);
      this.dog.body.enable = false;
      this.showToast('Koira lepää. 💤');
    }
  }

  onDogPickup(dog, pickup) {
    if (!this.dogState.active || this.dogState.mode !== 'fetch') return;
    // Assume coins 
    if (pickup.texture.key === 'tex_coin') {
      pickup.destroy();
      this.dogState.money += 1;
      const t = this.add.text(dog.x, dog.y-20, '+1💰', { fontSize:'12px', color:'#ffdd00', stroke:'#000', strokeThickness:2 }).setDepth(200);
      this.tweens.add({ targets:t, y:t.y-20, alpha:0, duration:600, onComplete:()=>t.destroy() });
    }
  }

  updateDog() {
    if (!this.dogState.active || !this.dog || !this.dog.active) return;
    
    const speed = this.dogState.stats.speed;
    const px = this.player.x, py = this.player.y;
    const dx = px - this.dog.x, dy = py - this.dog.y;
    const dist = Math.sqrt(dx*dx + dy*dy);

    let targetX = px;
    let shouldMove = false;
    let jump = false;

    if (this.dogState.mode === 'attack') {
        let best = null, bestDist = 250;
        const enemies = [this.zombies, this.slimes, this.wolves, this.birds, this.enemySoldiers];
        enemies.forEach(group => {
            group?.children?.iterate(e => {
                if (e && e.active) {
                    const edx = e.x - this.dog.x, edy = e.y - this.dog.y;
                    const d = Math.sqrt(edx*edx + edy*edy);
                    if (d < bestDist) { bestDist = d; best = e; }
                }
            });
        });

        if (best) {
            targetX = best.x;
            shouldMove = true;
            if (bestDist < 24) {
                 if (this.time.now > (this.dog._nextAttack||0)) {
                     this.dog._nextAttack = this.time.now + 800;
                     if (best.takeDamage) best.takeDamage(this.dogState.stats.damage);
                     else { best.destroy(); this.dropCoins(best.x, best.y, 1); }
                     // sfx
                     this.scene.scene.sound?.play?.('hit',{volume:0.5}); // or simple text
                     const t = this.add.text(best.x, best.y, 'BORK!', {fontSize:'10px', color:'#fff', stroke:'#000', strokeThickness:2});
                     this.time.delayedCall(300, ()=>t.destroy());
                 }
            } else if (best.y < this.dog.y - 20) {
              jump = true;
            }
        } else {
             if (dist > 60) shouldMove = true;
        }
    } else if (this.dogState.mode === 'fetch') {
        let best = null, bestDist = 300;
        this.pickups?.children?.iterate(c => {
             if (c && c.active && c.texture.key === 'tex_coin') {
                const cdx = c.x - this.dog.x, cdy = c.y - this.dog.y;
                const d = Math.sqrt(cdx*cdx + cdy*cdy);
                if (d < bestDist) { bestDist = d; best = c; }
             }
        });
        if (best) {
            targetX = best.x; shouldMove = true;
            if (best.y < this.dog.y - 20) jump = true;
        } else {
             if (dist > 60) shouldMove = true;
        }
    } else { // Follow
        if (dist > 50) shouldMove = true;
        if (py < this.dog.y - 40 && dist < 100) jump = true;
    }

    if (shouldMove) {
        const moveDx = targetX - this.dog.x;
        const moveDir = Math.sign(moveDx);
        if (Math.abs(moveDx) > 8) {
            this.dog.setVelocityX(moveDir * speed);
            this.dog.setFlipX(moveDir < 0);
        } else {
            this.dog.setVelocityX(0);
        }
    } else {
        this.dog.setVelocityX(0);
    }
    
    // Simple jump logic
    if ((jump || this.dog.body.blocked.left || this.dog.body.blocked.right) && this.dog.body.blocked.down) {
         this.dog.setVelocityY(-360);
    }
  }